# Day 4 · 실습 2. YOLO 모델 평가와 오류 분석

**연계 이론:** 이론 4 · **1.2~1.4** Precision·Recall·F1 / **3.1~3.2** TP·FP·FN / **5.2~5.5** 오류 분석

실습 1에서 저장한 YOLO 예측과 `assets/ground_truth_demo.json`의 정답 Box를 비교합니다. **`[직접 변경]`으로 표시된 세 가지 값만 수정**해 결과를 비교하세요.

> 정답 Box는 학습용으로 수동 표시한 **근사 좌표**입니다. 정식 성능 벤치마크가 아니며, 이 실습에서는 이미지별 `cup` 또는 `cat` **한 클래스만** 평가합니다.

In [ ]:
# 0. 준비 — 실습 1이 저장한 결과를 읽습니다 (추가 모델 추론 없음).
from pathlib import Path
import csv
import json
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
from PIL import Image

ROOT = next(
    (p for p in [Path.cwd(), *Path.cwd().parents]
     if (p / "assets" / "ground_truth_demo.json").is_file()),
    None,
)
if ROOT is None:
    raise FileNotFoundError(
        "assets/ground_truth_demo.json이 없습니다. 기존 assets/ 폴더의 위치를 확인하세요."
    )

ASSET_DIR = ROOT / "assets"
YOLO_FILE = ROOT / "outputs" / "yolo" / "yolo_detections.json"
OUT_DIR = ROOT / "outputs" / "analysis"
if not YOLO_FILE.is_file():
    raise FileNotFoundError(
        "실습 1의 outputs/yolo/yolo_detections.json이 없습니다. "
        "day4_02_yolo_detection.ipynb에서 먼저 추론 결과를 저장하세요."
    )

gt_meta = json.loads((ASSET_DIR / "ground_truth_demo.json").read_text(encoding="utf-8"))
yolo_record = json.loads(YOLO_FILE.read_text(encoding="utf-8"))
for name in gt_meta["images"]:
    if name not in yolo_record["images"]:
        raise ValueError(f"실습 1 결과에 {name} 이미지가 없습니다. 실습 1을 먼저 확인하세요.")
    if not (ASSET_DIR / name).is_file():
        raise FileNotFoundError(f"샘플 이미지가 없습니다: {ASSET_DIR / name}")

OUT_DIR.mkdir(parents=True, exist_ok=True)
print("준비 완료 | 평가 이미지:", ", ".join(gt_meta["images"]))
for name, info in gt_meta["images"].items():
    keys = sorted(yolo_record["images"][name]["thresholds"], key=float)
    print(f'  {name}: {info["target_class"]} / 저장된 Confidence 기준 {keys}')

## 1. TP·FP·FN과 Precision·Recall 계산 — 이론 4 · 1.2~1.4, 3.2

**평가용 IoU 기준**을 변경하면 같은 예측도 다르게 판정될 수 있습니다. 아래 세 값만 수정해 보세요.

In [ ]:
# [직접 변경 ①] 어떤 이미지의 결과를 그림으로 확인할까요?
IMAGE_NAME = "coffee.jpg"     # "coffee.jpg" 또는 "chelsea.jpg"

# [직접 변경 ②] 실습 1에서 저장한 Confidence 기준을 선택하세요.
CONF_THRESHOLD = "0.30"       # 기본값: "0.15", "0.30", "0.60"

# [직접 변경 ③] 정답 Box와 예측 Box의 일치 판정 기준입니다.
EVAL_IOU_THRESHOLD = 0.50     # 0.50과 0.75를 비교해 보세요.

assert IMAGE_NAME in gt_meta["images"], "IMAGE_NAME을 확인하세요."
available = list(yolo_record["images"][IMAGE_NAME]["thresholds"])
assert CONF_THRESHOLD in available, f"저장된 Confidence 기준 중 선택하세요: {available}"
assert 0 < EVAL_IOU_THRESHOLD <= 1, "IoU 기준은 0 초과, 1 이하여야 합니다."

In [ ]:
# 이론 4 · 3.2: 한 클래스의 예측을 Confidence 순서로 정답과 일대일 매칭합니다.
def box_iou(a, b):
    left, top = max(a[0], b[0]), max(a[1], b[1])
    right, bottom = min(a[2], b[2]), min(a[3], b[3])
    intersection = max(0, right-left) * max(0, bottom-top)
    area_a = max(0, a[2]-a[0]) * max(0, a[3]-a[1])
    area_b = max(0, b[2]-b[0]) * max(0, b[3]-b[1])
    union = area_a + area_b - intersection
    return intersection / union if union > 0 else 0.0

def evaluate(predictions, gt_boxes, target_class, threshold):
    # 수업용 GT가 붙은 한 클래스만 평가합니다. 다른 클래스는 계산에서 제외합니다.
    selected = sorted(
        (p for p in predictions if p["class_name"] == target_class),
        key=lambda p: p["confidence"], reverse=True,
    )
    remaining_gt = set(range(len(gt_boxes)))
    details = []
    for pred in selected:
        candidates = [(j, box_iou(pred["xyxy"], gt_boxes[j])) for j in remaining_gt]
        best_idx, best_iou = max(candidates, key=lambda pair: pair[1], default=(None, 0.0))
        is_tp = best_idx is not None and best_iou >= threshold
        if is_tp:
            remaining_gt.remove(best_idx)
        details.append({
            "xyxy": pred["xyxy"],
            "confidence": pred["confidence"],
            "status": "TP" if is_tp else "FP",
            "matched_gt": best_idx if is_tp else None,
            "iou": best_iou,
        })

    tp = sum(d["status"] == "TP" for d in details)
    fp, fn = len(details) - tp, len(remaining_gt)
    precision = tp / (tp + fp) if (tp + fp) else None
    recall = tp / (tp + fn) if (tp + fn) else None
    f1 = (2 * precision * recall / (precision + recall)
          if precision is not None and recall is not None and precision + recall > 0 else None)
    return {"TP": tp, "FP": fp, "FN": fn, "precision": precision,
            "recall": recall, "f1": f1, "details": details, "unmatched_gt": remaining_gt}

# 모든 이미지·Confidence 기준에 대해 동일한 IoU 기준으로 계산합니다.
analyses, rows = {}, []
for name, info in gt_meta["images"].items():
    analyses[name] = {}
    thresholds = yolo_record["images"][name]["thresholds"]
    for conf in sorted(thresholds, key=float):
        result = evaluate(thresholds[conf], info["boxes_xyxy"],
                          info["target_class"], EVAL_IOU_THRESHOLD)
        analyses[name][conf] = result
        rows.append({
            "image": name, "class": info["target_class"], "conf": conf,
            "iou_threshold": EVAL_IOU_THRESHOLD,
            **{k: result[k] for k in ["TP", "FP", "FN", "precision", "recall", "f1"]}
        })

fmt = lambda value: "N/A" if value is None else f"{value:.3f}"
print(f"\n평가 IoU 기준: {EVAL_IOU_THRESHOLD:.2f}  (수업용 근사 GT / 각 이미지의 대상 클래스만 평가)")
print(f'{"이미지":<13} {"conf":>5} {"TP":>3} {"FP":>3} {"FN":>3} {"Precision":>10} {"Recall":>8} {"F1":>7}')
for row in rows:
    print(f'{row["image"]:<13} {row["conf"]:>5} {row["TP"]:>3} {row["FP"]:>3} '
          f'{row["FN"]:>3} {fmt(row["precision"]):>10} {fmt(row["recall"]):>8} {fmt(row["f1"]):>7}')

csv_path = OUT_DIR / "detection_error_metrics.csv"
with csv_path.open("w", encoding="utf-8-sig", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=list(rows[0]))
    writer.writeheader()
    writer.writerows(rows)
print("결과 저장:", csv_path)

## 2. 예측 오류 시각화 — 이론 4 · 5.2~5.5

**점선**은 정답 Box, **초록 실선**은 TP, **빨간 실선**은 FP입니다. 매칭되지 않은 정답은 **FN**으로 표시합니다. 정답이 없는 다른 클래스의 예측은 표시하지 않습니다.

In [ ]:
# 1번에서 선택한 이미지·Confidence·평가 IoU 기준을 그대로 사용합니다.
info = gt_meta["images"][IMAGE_NAME]
result = analyses[IMAGE_NAME][CONF_THRESHOLD]
img = Image.open(ASSET_DIR / IMAGE_NAME).convert("RGB")
w, h = img.size

fig, ax = plt.subplots(figsize=(9, 6))
ax.imshow(img)

# Ground Truth: 매칭 여부에 따라 GT / FN 표기
for idx, (x1, y1, x2, y2) in enumerate(info["boxes_xyxy"]):
    missed = idx in result["unmatched_gt"]
    ax.add_patch(Rectangle((x1, y1), x2-x1, y2-y1,
                           fill=False, linestyle="--", linewidth=2.5,
                           edgecolor="orange" if missed else "yellow"))
    ax.text(x1, max(y1-6, 4), "GT: FN" if missed else "GT: matched",
            color="black", fontsize=10, fontweight="bold",
            bbox={"facecolor":"orange" if missed else "yellow", "alpha":0.8})

# Prediction: TP는 초록색, FP는 빨간색
for pred in result["details"]:
    x1, y1, x2, y2 = pred["xyxy"]
    color = "lime" if pred["status"] == "TP" else "red"
    ax.add_patch(Rectangle((x1, y1), x2-x1, y2-y1,
                           fill=False, linewidth=2.2, edgecolor=color))
    ax.text(x1, min(y2+13, h-4), f'{pred["status"]} ({pred["confidence"]:.2f})',
            color="white", fontsize=10, fontweight="bold",
            bbox={"facecolor":"darkgreen" if pred["status"] == "TP" else "darkred", "alpha":0.8})

ax.set_title(f'{IMAGE_NAME} | class={info["target_class"]} | '
             f'conf ≥ {CONF_THRESHOLD} | IoU ≥ {EVAL_IOU_THRESHOLD:.2f}\n'
             f'TP={result["TP"]}, FP={result["FP"]}, FN={result["FN"]}')
ax.set_xlim(0, w)
ax.set_ylim(h, 0)
ax.axis("off")
fig.tight_layout()
figure_path = OUT_DIR / (
    f'boxes_{Path(IMAGE_NAME).stem}_conf_{CONF_THRESHOLD.replace(".", "-")}'
    f'_iou_{EVAL_IOU_THRESHOLD:.2f}.png'
)
fig.savefig(figure_path, dpi=140, bbox_inches="tight")
plt.show()

print("그림 저장:", figure_path)
print("해석: FP는 정답에 매칭되지 않은 예측, FN은 끝까지 매칭되지 않은 정답입니다.")
print("비교: EVAL_IOU_THRESHOLD를 0.75로 바꾸고 1~2번 셀을 다시 실행해 보세요.")

## 3. 이번 실습 간단 요약

- **실습 1:** Confidence 기준에 따라 선택되는 탐지 결과를 비교했습니다.
- **실습 2:** 저장된 결과를 정답 Box와 비교하여 **TP·FP·FN, Precision·Recall·F1**을 계산했습니다.
- **해석:** 평가용 IoU 기준이 달라지면 TP·FP·FN 판정도 달라질 수 있습니다. `outputs/analysis/`에서 결과를 확인하세요.

※ 이번 결과는 **수업용 근사 정답을 이용한 평가 연습**이며, 공식 YOLO 성능 점수(AP·mAP)가 아닙니다.